In [ ]:
# InceptionTime SNR循环
# === 参数（请根据环境修改） ===
DATA_DIR = r'D:/users/zhongyuan/rf_datasets'   # 你的 .mat 文件文件夹
USE_LOG = True                # 是否对幅度取对数
WAVELET = 'db6'
WAVELET_LEVEL = 6
EXPECTED_LEN = 288             # DMRS 长度

# AWGN / Doppler 设置（这次 notebook 新增）
apply_awgn = True
apply_doppler = True
SNR_dB = 20                    # SNR in dB for AWGN
velocity_kmh = 120             # 多普勒目标速度（km/h）
fc = 5.9e9                     # 载频（Hz）用于计算多普勒
fs = 5e6                       # 采样率（Hz）

# 训练参数
BATCH_SIZE = 64
EPOCHS = 200
LR = 3e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5
PATIENCE = 5

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


# Imports and helper functions
import os, glob, h5py, math, time
import numpy as np
import pywt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, accuracy_score
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Subset
import torch.optim as optim
from datetime import datetime

# AWGN and Doppler helpers
def compute_doppler_shift(v_kmh, fc_hz):
    c = 3e8
    v = v_kmh / 3.6
    return fc_hz * v / c

def apply_doppler_shift(signal, fd_hz, fs_hz):
    t = np.arange(len(signal)) / fs_hz
    phase = np.exp(1j * 2 * np.pi * fd_hz * t)
    return signal * phase

def add_awgn(signal, snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    noise_power = signal_power / (10**(snr_db/10))
    
    # 关键修改：分别生成独立的高斯随机数
    noise_real = np.random.randn(*signal.shape)
    noise_imag = np.random.randn(*signal.shape)
    noise = np.sqrt(noise_power/2) * (noise_real + 1j*noise_imag)
    
    return signal + noise

# Wavelet + feature processing (db6, 6-level)
def process_dmrs_complex(sig_complex, use_log=False, wavelet='db6', level=6):
    # --------------------
    # 1. FFT 幅度谱
    # --------------------
    freq_sig = np.fft.fft(sig_complex)
    amp = np.abs(freq_sig)
    # 对称频谱，只取前半段
    amp = amp[:len(amp)//2]

    # --------------------
    # 2. 可选 log
    # --------------------
    if use_log:
        amp = np.log(amp + 1e-8)

    # --------------------
    # 3. 小波分解 + 低频置零 + 重构
    # --------------------
    coeffs = pywt.wavedec(amp, wavelet, level=level)
    coeffs[0] = np.zeros_like(coeffs[0])
    rec = pywt.waverec(coeffs, wavelet)
    rec = rec[:len(amp)]

    # --------------------
    # 4. 归一化
    # --------------------
    mu, sigma = rec.mean(), rec.std()
    if sigma < 1e-8:
        feat = (rec - mu).astype(np.float32)
    else:
        feat = ((rec - mu) / (sigma + 1e-8)).astype(np.float32)
    return feat

def moving_average(x, w=5):
    x = np.array(x)
    if len(x) == 0:
        return np.array([])   # 空的直接返回空数组
    if w <= 0:
        w = 1                 # 避免 np.ones(0)
    if len(x) < w:
        w = len(x)            # 窗口不能比序列长
    return np.convolve(x, np.ones(w), 'valid') / w

def plot_training_curves(fold_results, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(12,5))
    for i, res in enumerate(fold_results):
        plt.plot(moving_average(res['train_loss']), label=f'Fold{i+1} Train Loss')
        plt.plot(moving_average(res['val_loss']), label=f'Fold{i+1} Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('训练和验证Loss曲线')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'loss_curves.png'))
    plt.show()

def plot_grad_norms(avg_grad_norms, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(6,4))
    plt.bar(range(1, len(avg_grad_norms)+1), avg_grad_norms)
    plt.xlabel('Fold')
    plt.ylabel('平均梯度范数')
    plt.title('各Fold平均梯度范数')
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'avg_grad_norms.png'))
    plt.show()

    # 辅助函数绘图
def plot_and_save_cm(cm, save_folder, fold, dataset_type='Val'):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{dataset_type} CM Fold{fold}')
    plt.ylabel('Reference')
    plt.xlabel('Predicted')
    plt.savefig(os.path.join(save_folder, f'{dataset_type.lower()}_cm_fold{fold}.png'))
    plt.close()

def plot_acc_curves(fold_results, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(12,5))
    for i, res in enumerate(fold_results):
        plt.plot(res['train_acc'], label=f'Fold{i+1} TrainAcc')
        plt.plot(res['val_acc'], label=f'Fold{i+1} ValAcc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('训练集与验证集准确率曲线')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'accuracy_curves.png'))
    plt.close()

print('helpers ready')


# InceptionTime model definition (same as before)
import torch
import torch.nn as nn
import torch.nn.functional as F

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        bottleneck_channels = max(1, out_channels // 4)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.conv1 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=10, padding=5)
        self.conv2 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=20, padding=10)
        self.conv3 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=40, padding=20)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        self.convpool = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(4*out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_b = self.bottleneck(x)
        c1 = self.conv1(x_b)
        c2 = self.conv2(x_b)
        c3 = self.conv3(x_b)
        c4 = self.convpool(self.maxpool(x_b))

        # 保证长度一致，取最短长度
        min_len = min(c1.shape[-1], c2.shape[-1], c3.shape[-1], c4.shape[-1])
        c1 = c1[..., :min_len]
        c2 = c2[..., :min_len]
        c3 = c3[..., :min_len]
        c4 = c4[..., :min_len]

        out = torch.cat([c1, c2, c3, c4], dim=1)
        out = self.relu(self.bn(out))
        return out

class InceptionTime(nn.Module):
    def __init__(self, num_classes, in_channels=1, channels=32):
        super().__init__()
        self.b1 = InceptionBlock(in_channels, channels)
        self.b2 = InceptionBlock(4*channels, channels)
        self.b3 = InceptionBlock(4*channels, channels)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4*channels, num_classes)

    def forward(self, x):
        # 确保输入长度为偶数
        if x.shape[-1] % 2 == 1:
            x = x[..., :-1]

        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.gap(x).squeeze(-1)  # [batch, channels]
        out = self.fc(x)
        return out

print('Model defined')


# Training with K-fold and early stopping (single experiment config)
def evaluate_model_pt(model, dataloader, device, num_classes):
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            _, p = torch.max(out, 1)
            correct += (p == yb).sum().item()
            total += yb.size(0)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(p.cpu().numpy())
    acc = 100.0 * correct / total
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    return acc, cm

# Load DMRS .mat files and apply AWGN / Doppler on complex DMRS before processing
def load_features_with_channel(folder, expected_len=288, use_log=False, wavelet='db6', level=6,
                               apply_awgn_flag=False, snr_db=10, apply_doppler_flag=False, velocity_kmh=0, fc_hz=5.9e9, fs_hz=5e6, max_files=None):
    files = glob.glob(os.path.join(folder, '*.mat'))
    if max_files:
        files = files[:max_files]
    print('Found', len(files), 'MAT files in', folder)
    X, y = [], []
    for f in tqdm(files, desc='Loading .mat'):
        try:
            with h5py.File(f, 'r') as ff:
                if 'rfDataset' in ff:
                    ds = ff['rfDataset']
                    if 'dmrs' in ds:
                        dmrs = ds['dmrs'][:]
                    else:
                        # fallback: top-level dmrs
                        dmrs = ff.get('dmrs')[:]
                    txid = None
                    try:
                        if 'txID' in ds:
                            raw = ds['txID'][:].flatten()
                            if raw.dtype == 'uint16' or raw.dtype == 'uint8':
                                txid = ''.join(chr(int(c)) for c in raw if int(c) != 0)
                            else:
                                txid = str(raw)
                        else:
                            txid = os.path.basename(f)
                    except Exception:
                        txid = os.path.basename(f)
                else:
                    if 'dmrs' in ff:
                        dmrs = ff['dmrs'][:]
                        txid = os.path.basename(f)
                    else:
                        print('WARN: no dmrs found in', f)
                        continue
        except Exception as e:
            print('skip', f, '->', e)
            continue

        dmrs = np.array(dmrs)
        if dmrs.dtype.fields is not None and 'real' in dmrs.dtype.fields:
            real = dmrs['real']
            imag = dmrs['imag']
            dmrs = real + 1j*imag

        # ensure shape (num_samples, expected_len)
        if dmrs.ndim == 2 and dmrs.shape[0] == expected_len:
            dmrs = dmrs.T
        elif dmrs.ndim == 1 and dmrs.shape[0] == expected_len:
            dmrs = dmrs.reshape(1, -1)

        for i in range(dmrs.shape[0]):
            sig = dmrs[i]

            # 对每条信号进行功率归一化
            sig = sig / np.sqrt(np.mean(np.abs(sig)**2) + 1e-12)  # 避免除零

            # Apply Doppler ON COMPLEX DMRS
            if apply_doppler_flag:
                fd = compute_doppler_shift(velocity_kmh, fc_hz)
                sig = apply_doppler_shift(sig, fd, fs_hz)

            # Apply AWGN ON COMPLEX DMRS
            if apply_awgn_flag:
                sig = add_awgn(sig, snr_db)
            
            feat = process_dmrs_complex(sig, use_log=use_log, wavelet=wavelet, level=level)
            X.append(feat)
            y.append(txid)

    if len(X) == 0:
        raise RuntimeError('No samples loaded. Check DATA_DIR and .mat structure.')
    labels = sorted(list(set(y)))
    label_to_idx = {lab:i for i, lab in enumerate(labels)}
    y_idx = np.array([label_to_idx[lab] for lab in y], dtype=np.int64)
    X = np.stack(X, axis=0)
    print('Loaded features:', X.shape, 'labels:', y_idx.shape, 'num_classes:', len(labels))
    return X, y_idx, label_to_idx

# # Quick load (will warn if DATA_DIR missing)
# if not os.path.exists(DATA_DIR):
#     print('WARNING: DATA_DIR not found at', DATA_DIR)
# else:
#     X_all, y_all, label_to_idx = load_features_with_channel(
#         DATA_DIR, expected_len=EXPECTED_LEN, use_log=USE_LOG, wavelet=WAVELET, level=WAVELET_LEVEL,
#         apply_awgn_flag=apply_awgn, snr_db=SNR_dB, apply_doppler_flag=apply_doppler, velocity_kmh=velocity_kmh, fc_hz=fc, fs_hz=fs
#     )


def train_kfold(X, y, label_to_idx, device='cpu', batch_size=64, epochs=50, lr=3e-3,
                weight_decay=1e-4, n_splits=5, patience=8):

    import os, numpy as np, torch
    import torch.nn as nn, torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader, Subset
    from sklearn.model_selection import train_test_split, KFold
    from datetime import datetime
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm import tqdm

    # 确保 X shape = [num_samples, 1, length]
    X = np.array(X, dtype=np.float32)
    if X.ndim == 2:
        X = X[:, None, :]

    # train/test split
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )

    # 数据集
    full_dataset = TensorDataset(torch.tensor(X_trainval, dtype=torch.float32),
                                 torch.tensor(y_trainval, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.long))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 创建保存文件夹和 results.txt
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    script_name = "LTE-V_LED"
    fd = int(compute_doppler_shift(velocity_kmh, fc))
    save_dir = f"{timestamp}_{script_name}_SNR{snr_db}dB_fd{fd}_classes_{len(label_to_idx)}_CNN"
    save_folder = os.path.join(os.getcwd(), "training_results", save_dir)
    os.makedirs(save_folder, exist_ok=True)
    results_file = os.path.join(save_folder, "results.txt")

    # 保存实验参数
    with open(results_file, 'w') as f:
        f.write("=== Experiment Parameters ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"SNR_dB: {snr_db}, Apply AWGN: {apply_awgn}\n")
        f.write(f"Velocity_kmh: {velocity_kmh}, Apply Doppler: {apply_doppler}\n")
        f.write(f"Carrier frequency: {fc} Hz, Sampling rate: {fs} Hz\n")
        f.write(f"Expected DMRS length: {EXPECTED_LEN}\n")
        f.write(f"Wavelet: {WAVELET}, Level: {WAVELET_LEVEL}, Use Log: {USE_LOG}\n")
        f.write(f"Training params: batch_size={batch_size}, epochs={epochs}, lr={lr}, weight_decay={weight_decay}, patience={patience}\n")
        f.write(f"Number of classes: {len(label_to_idx)}\n")
        f.write("============================\n\n")

    # --- KFold ---
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(full_dataset))

    fold_results = []
    val_scores, test_scores = [], []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(indices)):
        print(f'\n=== Fold {fold+1} ===')
        tr_sub, va_sub = Subset(full_dataset, tr_idx), Subset(full_dataset, va_idx)
        tr_loader = DataLoader(tr_sub, batch_size=batch_size, shuffle=True)
        va_loader = DataLoader(va_sub, batch_size=batch_size, shuffle=False)

        model = InceptionTime(num_classes=len(label_to_idx)).to(device)
        crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        sched = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

        best_val, best_wts, patience_cnt = 0.0, None, 0
        train_losses, val_losses, grad_norms = [], [], []
        train_acc_list, val_acc_list = [], []

        for epoch in range(epochs):
            model.train()
            running_loss, correct, total, batch_grads = 0.0, 0, 0, []

            for xb, yb in tqdm(tr_loader, desc=f"Fold{fold+1} Epoch{epoch+1}", leave=False):
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                out = model(xb)
                loss = crit(out, yb)
                loss.backward()

                # grad norm
                total_norm = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        total_norm += p.grad.data.norm(2).item()**2
                total_norm = total_norm**0.5
                batch_grads.append(total_norm)

                opt.step()
                running_loss += loss.item()
                _, p = torch.max(out, 1)
                correct += (p == yb).sum().item()
                total += yb.size(0)

            sched.step()
            train_loss_epoch = running_loss / len(tr_loader)
            train_acc_epoch = 100.0 * correct / total
            train_losses.append(train_loss_epoch)
            grad_norms.append(np.mean(batch_grads) if batch_grads else 0.0)
            train_acc_list.append(train_acc_epoch)

            # validation
            model.eval()
            vloss, vcorrect, vtotal = 0.0, 0, 0
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    out = model(xb)
                    loss = crit(out, yb)
                    vloss += loss.item()
                    _, p = torch.max(out, 1)
                    vcorrect += (p == yb).sum().item()
                    vtotal += yb.size(0)
            vloss /= len(va_loader)
            val_acc_epoch = 100.0 * vcorrect / vtotal
            val_losses.append(vloss)
            val_acc_list.append(val_acc_epoch)

            print(f"Epoch {epoch+1}/{epochs} | TrainAcc={train_acc_epoch:.2f}% | ValAcc={val_acc_epoch:.2f}% | TrainLoss={train_loss_epoch:.4f} | ValLoss={vloss:.4f}")
            with open(results_file, 'a') as f:
                f.write(f"Fold{fold+1} Epoch{epoch+1} TrainAcc={train_acc_epoch:.2f}% ValAcc={val_acc_epoch:.2f}% TrainLoss={train_loss_epoch:.4f} ValLoss={vloss:.4f}\n")

            # early stopping
            if val_acc_epoch > best_val + 0.01:
                best_val = val_acc_epoch
                best_wts = model.state_dict()
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience:
                    print("Early stopping.")
                    break

        # restore best weights
        if best_wts is not None:
            model.load_state_dict(best_wts)

        # 评估
        train_acc_fold, train_cm = evaluate_model_pt(model, tr_loader, device, len(label_to_idx))
        val_acc_fold, val_cm = evaluate_model_pt(model, va_loader, device, len(label_to_idx))
        test_acc_fold, test_cm = evaluate_model_pt(model, test_loader, device, len(label_to_idx))

        print(f"Fold{fold+1} Summary | TrainAcc={train_acc_fold:.2f}% | ValAcc={val_acc_fold:.2f}% | TestAcc={test_acc_fold:.2f}%")
        with open(results_file, 'a') as f:
            f.write(f"Fold{fold+1} Final | TrainAcc={train_acc_fold:.2f}% ValAcc={val_acc_fold:.2f}% TestAcc={test_acc_fold:.2f}%\n")

        # 保存混淆矩阵
        np.save(os.path.join(save_folder, f'train_cm_fold{fold+1}.npy'), train_cm)
        np.save(os.path.join(save_folder, f'val_cm_fold{fold+1}.npy'), val_cm)
        np.save(os.path.join(save_folder, f'test_cm_fold{fold+1}.npy'), test_cm)
        plot_and_save_cm(train_cm, save_folder, fold+1, 'Train')
        plot_and_save_cm(val_cm, save_folder, fold+1, 'Val')
        plot_and_save_cm(test_cm, save_folder, fold+1, 'Test')

        # 保存当前 fold 的训练/验证曲线
        fold_save_folder = os.path.join(save_folder, f'Fold{fold+1}')
        os.makedirs(fold_save_folder, exist_ok=True)
        plot_training_curves([{'train_loss': train_losses, 'val_loss': val_losses}], fold_save_folder)
        plot_acc_curves([{'train_acc': train_acc_list, 'val_acc': val_acc_list}], fold_save_folder)
        plot_grad_norms([np.mean(grad_norms)], fold_save_folder)

        # 保存 fold 结果
        fold_results.append({
            'train_loss': train_losses,
            'val_loss': val_losses,
            'grad_norms': grad_norms,
            'train_acc': train_acc_list,
            'val_acc': val_acc_list
        })

        val_scores.append(val_acc_fold)
        test_scores.append(test_acc_fold)

    # 总结
    print("\n=== Overall Summary ===")
    print(f"Val Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}")
    print(f"Test Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}")
    with open(results_file, 'a') as f:
        f.write(f"\n=== Overall Summary ===\nVal Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}\nTest Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}\n")

    # 最终整体图（所有 fold）
    plot_training_curves(fold_results, save_folder)
    plot_acc_curves(fold_results, save_folder)
    plot_grad_norms([np.mean(fr['grad_norms']) for fr in fold_results], save_folder)

    return save_folder


# ================== SNR 循环 ==================
snr_list = range(20, -45, -5)
for snr_db in snr_list:  # 20, 15, ..., -40 dB
    print(f"\n\n=== Running experiment for SNR={snr_db} dB ===")

    # 重新加载特征，加上当前 SNR 的 AWGN
    X_snr, y_snr, label_to_idx_snr = load_features_with_channel(
        DATA_DIR,
        expected_len=EXPECTED_LEN,
        use_log=USE_LOG,
        wavelet=WAVELET,
        level=WAVELET_LEVEL,
        apply_awgn_flag=apply_awgn,
        snr_db=snr_db,
        apply_doppler_flag=apply_doppler,
        velocity_kmh=velocity_kmh,
        fc_hz=fc,
        fs_hz=fs
        )

    # 调用训练函数
    save_folder = train_kfold(
        X_snr, y_snr, label_to_idx_snr,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        n_splits=N_SPLITS,
        patience=PATIENCE
    )
    print(f"SNR {snr_db} dB experiment results saved to {save_folder}")


# LED-RFF 复现（含 AWGN 与 多普勒 处理）

**文件名：`LTE_LED_Doppler_AWGN.ipynb`**

这个 notebook 在之前 LED-RFF 复现的基础上，**在复数 DMRS 上增加了 AWGN 加噪与多普勒频移处理**，并将这些处理放在小波分解之前（符合物理意义）。

主要流程：

1. 从 `.mat` 文件加载 `rfDataset.dmrs`（复数 DMRS）
2. （可选）对复数 DMRS 应用 **多普勒频移**
3. （可选）对复数 DMRS 添加 **AWGN**
4. 取幅度 → （可选 log）→ db6 小波分解（6 层）→ 将 A6 置零 → 重构 → 归一化
5. 使用 InceptionTime 分类器训练与评估（含 K-fold、早停）


In [ ]:
# InceptionTime 单次版
# === 参数（请根据环境修改） ===
DATA_DIR = r'E:/rf_datasets'   # 你的 .mat 文件文件夹
USE_LOG = False                # 是否对幅度取对数
WAVELET = 'db6'
WAVELET_LEVEL = 6
EXPECTED_LEN = 288             # DMRS 长度

# AWGN / Doppler 设置（这次 notebook 新增）
apply_awgn = True
apply_doppler = True
SNR_dB = 20                    # SNR in dB for AWGN
velocity_kmh = 120             # 多普勒目标速度（km/h）
fc = 5.9e9                     # 载频（Hz）用于计算多普勒
fs = 5e6                       # 采样率（Hz）

# 训练参数
BATCH_SIZE = 64
EPOCHS = 200
LR = 3e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5
PATIENCE = 5

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


# Imports and helper functions
import os, glob, h5py, math, time
import numpy as np
import pywt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, accuracy_score
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Subset
import torch.optim as optim
from datetime import datetime

# AWGN and Doppler helpers
def compute_doppler_shift(v_kmh, fc_hz):
    c = 3e8
    v = v_kmh / 3.6
    return fc_hz * v / c

def apply_doppler_shift(signal, fd_hz, fs_hz):
    t = np.arange(len(signal)) / fs_hz
    phase = np.exp(1j * 2 * np.pi * fd_hz * t)
    return signal * phase

def add_awgn(signal, snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    noise_power = signal_power / (10**(snr_db/10))
    
    # 关键修改：分别生成独立的高斯随机数
    noise_real = np.random.randn(*signal.shape)
    noise_imag = np.random.randn(*signal.shape)
    noise = np.sqrt(noise_power/2) * (noise_real + 1j*noise_imag)
    
    return signal + noise

# Wavelet + feature processing (db6, 6-level)
def process_dmrs_complex(sig_complex, use_log=False, wavelet='db6', level=6):
    # --------------------
    # 1. FFT 幅度谱
    # --------------------
    freq_sig = np.fft.fft(sig_complex)
    amp = np.abs(freq_sig)
    # 对称频谱，只取前半段
    amp = amp[:len(amp)//2]

    # --------------------
    # 2. 可选 log
    # --------------------
    if use_log:
        amp = np.log(amp + 1e-8)

    # --------------------
    # 3. 小波分解 + 低频置零 + 重构
    # --------------------
    coeffs = pywt.wavedec(amp, wavelet, level=level)
    coeffs[0] = np.zeros_like(coeffs[0])
    rec = pywt.waverec(coeffs, wavelet)
    rec = rec[:len(amp)]

    # --------------------
    # 4. 归一化
    # --------------------
    mu, sigma = rec.mean(), rec.std()
    if sigma < 1e-8:
        feat = (rec - mu).astype(np.float32)
    else:
        feat = ((rec - mu) / (sigma + 1e-8)).astype(np.float32)
    return feat

def moving_average(x, w=5):
    x = np.array(x)
    if len(x) == 0:
        return np.array([])   # 空的直接返回空数组
    if w <= 0:
        w = 1                 # 避免 np.ones(0)
    if len(x) < w:
        w = len(x)            # 窗口不能比序列长
    return np.convolve(x, np.ones(w), 'valid') / w

def plot_training_curves(fold_results, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(12,5))
    for i, res in enumerate(fold_results):
        plt.plot(moving_average(res['train_loss']), label=f'Fold{i+1} Train Loss')
        plt.plot(moving_average(res['val_loss']), label=f'Fold{i+1} Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('训练和验证Loss曲线')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'loss_curves.png'))
    plt.show()

def plot_grad_norms(avg_grad_norms, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(6,4))
    plt.bar(range(1, len(avg_grad_norms)+1), avg_grad_norms)
    plt.xlabel('Fold')
    plt.ylabel('平均梯度范数')
    plt.title('各Fold平均梯度范数')
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'avg_grad_norms.png'))
    plt.show()

    # 辅助函数绘图
def plot_and_save_cm(cm, save_folder, fold, dataset_type='Val'):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{dataset_type} CM Fold{fold}')
    plt.ylabel('Reference')
    plt.xlabel('Predicted')
    plt.savefig(os.path.join(save_folder, f'{dataset_type.lower()}_cm_fold{fold}.png'))
    plt.close()

def plot_acc_curves(fold_results, save_folder):
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
    plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题
    plt.figure(figsize=(12,5))
    for i, res in enumerate(fold_results):
        plt.plot(res['train_acc'], label=f'Fold{i+1} TrainAcc')
        plt.plot(res['val_acc'], label=f'Fold{i+1} ValAcc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('训练集与验证集准确率曲线')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_folder, 'accuracy_curves.png'))
    plt.close()

print('helpers ready')


# InceptionTime model definition (same as before)
import torch
import torch.nn as nn
import torch.nn.functional as F

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        bottleneck_channels = max(1, out_channels // 4)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.conv1 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=10, padding=5)
        self.conv2 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=20, padding=10)
        self.conv3 = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=40, padding=20)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        self.convpool = nn.Conv1d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(4*out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_b = self.bottleneck(x)
        c1 = self.conv1(x_b)
        c2 = self.conv2(x_b)
        c3 = self.conv3(x_b)
        c4 = self.convpool(self.maxpool(x_b))

        # 保证长度一致，取最短长度
        min_len = min(c1.shape[-1], c2.shape[-1], c3.shape[-1], c4.shape[-1])
        c1 = c1[..., :min_len]
        c2 = c2[..., :min_len]
        c3 = c3[..., :min_len]
        c4 = c4[..., :min_len]

        out = torch.cat([c1, c2, c3, c4], dim=1)
        out = self.relu(self.bn(out))
        return out

class InceptionTime(nn.Module):
    def __init__(self, num_classes, in_channels=1, channels=32):
        super().__init__()
        self.b1 = InceptionBlock(in_channels, channels)
        self.b2 = InceptionBlock(4*channels, channels)
        self.b3 = InceptionBlock(4*channels, channels)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4*channels, num_classes)

    def forward(self, x):
        # 确保输入长度为偶数
        if x.shape[-1] % 2 == 1:
            x = x[..., :-1]

        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.gap(x).squeeze(-1)  # [batch, channels]
        out = self.fc(x)
        return out

print('Model defined')


# Training with K-fold and early stopping (single experiment config)
def evaluate_model_pt(model, dataloader, device, num_classes):
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            _, p = torch.max(out, 1)
            correct += (p == yb).sum().item()
            total += yb.size(0)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(p.cpu().numpy())
    acc = 100.0 * correct / total
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    return acc, cm

# Load DMRS .mat files and apply AWGN / Doppler on complex DMRS before processing
def load_features_with_channel(folder, expected_len=288, use_log=False, wavelet='db6', level=6,
                               apply_awgn_flag=False, snr_db=10, apply_doppler_flag=False, velocity_kmh=0, fc_hz=5.9e9, fs_hz=5e6, max_files=None):
    files = glob.glob(os.path.join(folder, '*.mat'))
    if max_files:
        files = files[:max_files]
    print('Found', len(files), 'MAT files in', folder)
    X, y = [], []
    for f in tqdm(files, desc='Loading .mat'):
        try:
            with h5py.File(f, 'r') as ff:
                if 'rfDataset' in ff:
                    ds = ff['rfDataset']
                    if 'dmrs' in ds:
                        dmrs = ds['dmrs'][:]
                    else:
                        # fallback: top-level dmrs
                        dmrs = ff.get('dmrs')[:]
                    txid = None
                    try:
                        if 'txID' in ds:
                            raw = ds['txID'][:].flatten()
                            if raw.dtype == 'uint16' or raw.dtype == 'uint8':
                                txid = ''.join(chr(int(c)) for c in raw if int(c) != 0)
                            else:
                                txid = str(raw)
                        else:
                            txid = os.path.basename(f)
                    except Exception:
                        txid = os.path.basename(f)
                else:
                    if 'dmrs' in ff:
                        dmrs = ff['dmrs'][:]
                        txid = os.path.basename(f)
                    else:
                        print('WARN: no dmrs found in', f)
                        continue
        except Exception as e:
            print('skip', f, '->', e)
            continue

        dmrs = np.array(dmrs)
        if dmrs.dtype.fields is not None and 'real' in dmrs.dtype.fields:
            real = dmrs['real']
            imag = dmrs['imag']
            dmrs = real + 1j*imag

        # ensure shape (num_samples, expected_len)
        if dmrs.ndim == 2 and dmrs.shape[0] == expected_len:
            dmrs = dmrs.T
        elif dmrs.ndim == 1 and dmrs.shape[0] == expected_len:
            dmrs = dmrs.reshape(1, -1)

        for i in range(dmrs.shape[0]):
            sig = dmrs[i]

            # === step1: 原始信号功率归一化 ===
            sig = sig / (np.sqrt(np.mean(np.abs(sig)**2)) + 1e-12)

            # === step2: Doppler （只改变相位，不改变功率） ===
            if apply_doppler:
                sig = apply_doppler_shift(sig, fd, fs)

            # === step3: AWGN（严格按照 SNR 产生噪声） ===
            if apply_awgn:
                sig = add_awgn(sig, snr_db)

            feat = process_dmrs_complex(sig, use_log=use_log, wavelet=wavelet, level=level)
            X.append(feat)
            y.append(txid)

    if len(X) == 0:
        raise RuntimeError('No samples loaded. Check DATA_DIR and .mat structure.')
    labels = sorted(list(set(y)))
    label_to_idx = {lab:i for i, lab in enumerate(labels)}
    y_idx = np.array([label_to_idx[lab] for lab in y], dtype=np.int64)
    X = np.stack(X, axis=0)
    print('Loaded features:', X.shape, 'labels:', y_idx.shape, 'num_classes:', len(labels))
    return X, y_idx, label_to_idx

# Quick load (will warn if DATA_DIR missing)
if not os.path.exists(DATA_DIR):
    print('WARNING: DATA_DIR not found at', DATA_DIR)
else:
    X_all, y_all, label_to_idx = load_features_with_channel(
        DATA_DIR, expected_len=EXPECTED_LEN, use_log=USE_LOG, wavelet=WAVELET, level=WAVELET_LEVEL,
        apply_awgn_flag=apply_awgn, snr_db=SNR_dB, apply_doppler_flag=apply_doppler, velocity_kmh=velocity_kmh, fc_hz=fc, fs_hz=fs
    )


def train_kfold(X, y, label_to_idx, device='cpu', batch_size=64, epochs=50, lr=3e-3,
                weight_decay=1e-4, n_splits=5, patience=8):

    import os, numpy as np, torch
    import torch.nn as nn, torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader, Subset
    from sklearn.model_selection import train_test_split, KFold
    from datetime import datetime
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm import tqdm

    # 确保 X shape = [num_samples, 1, length]
    X = np.array(X, dtype=np.float32)
    if X.ndim == 2:
        X = X[:, None, :]

    # train/test split
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )

    # 数据集
    full_dataset = TensorDataset(torch.tensor(X_trainval, dtype=torch.float32),
                                 torch.tensor(y_trainval, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                 torch.tensor(y_test, dtype=torch.long))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 创建保存文件夹和 results.txt
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    script_name = "LTE-V_LED"
    fd = int(compute_doppler_shift(velocity_kmh, fc))
    save_dir = f"{timestamp}_{script_name}_SNR{SNR_dB}dB_fd{fd}_classes_{len(label_to_idx)}_CNN"
    save_folder = os.path.join(os.getcwd(), "training_results", save_dir)
    os.makedirs(save_folder, exist_ok=True)
    results_file = os.path.join(save_folder, "results.txt")

    # 保存实验参数
    with open(results_file, 'w') as f:
        f.write("=== Experiment Parameters ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"SNR_dB: {SNR_dB}, Apply AWGN: {apply_awgn}\n")
        f.write(f"Velocity_kmh: {velocity_kmh}, Apply Doppler: {apply_doppler}\n")
        f.write(f"Carrier frequency: {fc} Hz, Sampling rate: {fs} Hz\n")
        f.write(f"Expected DMRS length: {EXPECTED_LEN}\n")
        f.write(f"Wavelet: {WAVELET}, Level: {WAVELET_LEVEL}, Use Log: {USE_LOG}\n")
        f.write(f"Training params: batch_size={batch_size}, epochs={epochs}, lr={lr}, weight_decay={weight_decay}, patience={patience}\n")
        f.write(f"Number of classes: {len(label_to_idx)}\n")
        f.write("============================\n\n")

    # --- KFold ---
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(full_dataset))

    fold_results = []
    val_scores, test_scores = [], []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(indices)):
        print(f'\n=== Fold {fold+1} ===')
        tr_sub, va_sub = Subset(full_dataset, tr_idx), Subset(full_dataset, va_idx)
        tr_loader = DataLoader(tr_sub, batch_size=batch_size, shuffle=True)
        va_loader = DataLoader(va_sub, batch_size=batch_size, shuffle=False)

        model = InceptionTime(num_classes=len(label_to_idx)).to(device)
        crit = nn.CrossEntropyLoss()
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        sched = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

        best_val, best_wts, patience_cnt = 0.0, None, 0
        train_losses, val_losses, grad_norms = [], [], []
        train_acc_list, val_acc_list = [], []

        for epoch in range(epochs):
            model.train()
            running_loss, correct, total, batch_grads = 0.0, 0, 0, []

            for xb, yb in tqdm(tr_loader, desc=f"Fold{fold+1} Epoch{epoch+1}", leave=False):
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                out = model(xb)
                loss = crit(out, yb)
                loss.backward()

                # grad norm
                total_norm = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        total_norm += p.grad.data.norm(2).item()**2
                total_norm = total_norm**0.5
                batch_grads.append(total_norm)

                opt.step()
                running_loss += loss.item()
                _, p = torch.max(out, 1)
                correct += (p == yb).sum().item()
                total += yb.size(0)

            sched.step()
            train_loss_epoch = running_loss / len(tr_loader)
            train_acc_epoch = 100.0 * correct / total
            train_losses.append(train_loss_epoch)
            grad_norms.append(np.mean(batch_grads) if batch_grads else 0.0)
            train_acc_list.append(train_acc_epoch)

            # validation
            model.eval()
            vloss, vcorrect, vtotal = 0.0, 0, 0
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    out = model(xb)
                    loss = crit(out, yb)
                    vloss += loss.item()
                    _, p = torch.max(out, 1)
                    vcorrect += (p == yb).sum().item()
                    vtotal += yb.size(0)
            vloss /= len(va_loader)
            val_acc_epoch = 100.0 * vcorrect / vtotal
            val_losses.append(vloss)
            val_acc_list.append(val_acc_epoch)

            print(f"Epoch {epoch+1}/{epochs} | TrainAcc={train_acc_epoch:.2f}% | ValAcc={val_acc_epoch:.2f}% | TrainLoss={train_loss_epoch:.4f} | ValLoss={vloss:.4f}")
            with open(results_file, 'a') as f:
                f.write(f"Fold{fold+1} Epoch{epoch+1} TrainAcc={train_acc_epoch:.2f}% ValAcc={val_acc_epoch:.2f}% TrainLoss={train_loss_epoch:.4f} ValLoss={vloss:.4f}\n")

            # early stopping
            if val_acc_epoch > best_val + 0.01:
                best_val = val_acc_epoch
                best_wts = model.state_dict()
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience:
                    print("Early stopping.")
                    break

        # restore best weights
        if best_wts is not None:
            model.load_state_dict(best_wts)

        # 评估
        train_acc_fold, train_cm = evaluate_model_pt(model, tr_loader, device, len(label_to_idx))
        val_acc_fold, val_cm = evaluate_model_pt(model, va_loader, device, len(label_to_idx))
        test_acc_fold, test_cm = evaluate_model_pt(model, test_loader, device, len(label_to_idx))

        print(f"Fold{fold+1} Summary | TrainAcc={train_acc_fold:.2f}% | ValAcc={val_acc_fold:.2f}% | TestAcc={test_acc_fold:.2f}%")
        with open(results_file, 'a') as f:
            f.write(f"Fold{fold+1} Final | TrainAcc={train_acc_fold:.2f}% ValAcc={val_acc_fold:.2f}% TestAcc={test_acc_fold:.2f}%\n")

        # 保存混淆矩阵
        np.save(os.path.join(save_folder, f'train_cm_fold{fold+1}.npy'), train_cm)
        np.save(os.path.join(save_folder, f'val_cm_fold{fold+1}.npy'), val_cm)
        np.save(os.path.join(save_folder, f'test_cm_fold{fold+1}.npy'), test_cm)
        plot_and_save_cm(train_cm, save_folder, fold+1, 'Train')
        plot_and_save_cm(val_cm, save_folder, fold+1, 'Val')
        plot_and_save_cm(test_cm, save_folder, fold+1, 'Test')

        # 保存当前 fold 的训练/验证曲线
        fold_save_folder = os.path.join(save_folder, f'Fold{fold+1}')
        os.makedirs(fold_save_folder, exist_ok=True)
        plot_training_curves([{'train_loss': train_losses, 'val_loss': val_losses}], fold_save_folder)
        plot_acc_curves([{'train_acc': train_acc_list, 'val_acc': val_acc_list}], fold_save_folder)
        plot_grad_norms([np.mean(grad_norms)], fold_save_folder)

        # 保存 fold 结果
        fold_results.append({
            'train_loss': train_losses,
            'val_loss': val_losses,
            'grad_norms': grad_norms,
            'train_acc': train_acc_list,
            'val_acc': val_acc_list
        })

        val_scores.append(val_acc_fold)
        test_scores.append(test_acc_fold)

    # 总结
    print("\n=== Overall Summary ===")
    print(f"Val Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}")
    print(f"Test Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}")
    with open(results_file, 'a') as f:
        f.write(f"\n=== Overall Summary ===\nVal Acc: {np.mean(val_scores):.2f} ± {np.std(val_scores):.2f}\nTest Acc: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f}\n")

    # 最终整体图（所有 fold）
    plot_training_curves(fold_results, save_folder)
    plot_acc_curves(fold_results, save_folder)
    plot_grad_norms([np.mean(fr['grad_norms']) for fr in fold_results], save_folder)

    return save_folder


# Run training if data was loaded
if 'X_all' in globals():
    save_folder = train_kfold(X_all, y_all, label_to_idx, device=DEVICE, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, n_splits=N_SPLITS, patience=PATIENCE)
    print('Results saved to', save_folder)
else:
    print('No X_all loaded. Run the load cell first.')